<h2>BMIF201: inference from genetic variation</h2>

Complete this activity in groups of 2-3. All members of each group should type their own solutions, but the solutions can be exactly identical. Feel free to ask other groups, the TAs, or the professor for assistance. It is OK if you do not complete it.

We use $4N$ generations to approach equilibrium. The simulations are short, so the results will be approximate.

Fill in the lines marked `# TODO`, replacing `...` with your code. The simulation loops and plotting code are supplied.

In class, we mostly derived results of the form, "what happens under this evolutionary process?" This activity uses simulations to ask, "how can we reconstruct or estimate that process from the data that it generates?" 

There are 12 problems, most of which ask you to implement 1-2 lines of code and think about the results.

<h5>Problem 1</h5>

Implement the two lines marked `# TODO`: the probability after mutation and the number of carriers after drift. Check that the assertions pass in the following code block.

In [ ]:
import numpy as np

class WrightFisher:
    def __init__(self, mu, N):
        self.mu = mu
        self.N = N  # There are 2N copies of each site.

    def next_generation(self, allele_frequencies):
        # TODO: probability after mutation. Do not neglect back-mutation.
        expected_frequency = ...
        # TODO: number of carriers after drift. Use np.random.binomial(...).
        num_carriers = ...
        return num_carriers/(2*self.N)

    def evolve(self, allele_frequencies, generations):
        for _ in range(generations):
            allele_frequencies = self.next_generation(allele_frequencies)
        return allele_frequencies

burn_in = 4  # generations per N

In [ ]:
num_sites = 5
model = WrightFisher(mu=1e-1, N=100)

np.random.seed(123)
generation_zero_frequency = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
generation_one_frequency = model.next_generation(generation_zero_frequency)
assert generation_one_frequency.shape == generation_zero_frequency.shape
assert np.all((generation_one_frequency >= 0) & (generation_one_frequency <= 1))
assert np.allclose(2*model.N*generation_one_frequency,
                   np.rint(2*model.N*generation_one_frequency))
print('Success!')

<h5> Problem 2 </h5>

In lecture we derived the mean heterozygosity at equilibrium:
$$E(2p(1-p))\approx 4N\mu$$
Show that with a population size of 100, the approximation is good for small $\mu$. Find the approximate value of $\mu$ at which the approximation breaks down. The code below starts with all frequencies at zero and runs for $4N$ generations.

Fill in the one marked line defining `H`, then run the supplied simulation and plotting cells.

In [ ]:
def heterozygosity(allele_frequency):
    # TODO: mean heterozygosity across sites, for each replicate.
    H = ...
    return H

In [ ]:
import matplotlib.pyplot as plt

np.random.seed(201)
population_size = 100
num_sites = 3000
mutation_rate = np.array([0.002, 0.005,
                          0.01, 0.02, 0.05, 0.1, 1, 10]) / population_size
mean_heterozygosity = []
for mu in mutation_rate:
    model = WrightFisher(mu=mu, N=population_size)
    allele_frequency = model.evolve(np.zeros(num_sites), burn_in*population_size)
    mean_heterozygosity.append(heterozygosity(allele_frequency))

In [ ]:
# Supplied plotting code.
theta = 4*population_size*mutation_rate
plt.figure(figsize=(8, 5))
plt.plot(mutation_rate, theta, label=r'$4N\mu$')
plt.scatter(mutation_rate, mean_heterozygosity, color='red', label='Simulated')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Mutation rate')
plt.ylabel('Mean heterozygosity')
plt.legend()
plt.tight_layout()
plt.show()
print('Mutation rate    4Nmu     Mean heterozygosity')
for mu, expected, observed in zip(mutation_rate, theta, mean_heterozygosity):
    print(f'{mu:13.2g}  {expected:7.4f}  {observed:7.4f}')

<h5> Problem 3 </h5>

The number of segregating sites in a population is the number of sites at which both alleles are present. In a large population, the number of segregating sites grows, and most are rare. The code below simulates populations of size $N\in \{10, 50, 250\}$, with $0 < \mu << 1/N$. Calculate the mean heterozygosity *of segregating sites*, $E(2p(1-p)|0<p<1)$, and compare with the mean heterozygosity across all sites. Discuss with your partner: if the mean heterozygosity increases with $N$, why does the mean heterozygosity of segregating sites decrease? (Hint: consider the site frequency spectrum in Problem 4).

Fill in the two marked lines for `mean_H` and `mean_H_segregating`. Run the supplied plotting cell.

In [ ]:
np.random.seed(202)
population_sizes = [10, 50, 250]
num_sites = 5000
mu = 5e-5
heterozygosity_all = []
heterozygosity_segregating = []
fraction_segregating = []
for N in population_sizes:
    model = WrightFisher(mu=mu, N=N)
    allele_frequency = model.evolve(np.zeros(num_sites), burn_in*N)
    segregating = (allele_frequency > 0) & (allele_frequency < 1)
    H = 2*allele_frequency*(1-allele_frequency)
    # TODO: mean heterozygosity across all sites.
    mean_H = ...
    # TODO: mean heterozygosity across segregating sites.
    mean_H_segregating = ...
    heterozygosity_all.append(mean_H)
    heterozygosity_segregating.append(mean_H_segregating)
    fraction_segregating.append(segregating.mean())

In [ ]:
# Supplied plotting code.
print('N      H: all sites   H: segregating sites   Fraction segregating')
for row in zip(population_sizes, heterozygosity_all,
               heterozygosity_segregating, fraction_segregating):
    print(f'{row[0]:4d}   {row[1]:12.5f}   {row[2]:20.4f}   {row[3]:20.4f}')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(population_sizes, heterozygosity_all, 'o-')
axes[0].plot(population_sizes, 4*np.array(population_sizes)*mu,
             '--', color='gray', label=r'$4N\mu$')
axes[0].set_ylabel('Mean heterozygosity: all sites')
axes[0].legend()
axes[1].plot(population_sizes, heterozygosity_segregating, 'o-')
axes[1].set_ylabel('Mean heterozygosity: segregating sites')
for ax in axes:
    ax.set_xscale('log')
    ax.set_xlabel('Population size, N')
plt.tight_layout()
plt.show()

<h5>Problem 4</h5>

The site frequency spectrum counts how many sites have a derived allele observed $i$ times. Write this count as $k_i$. For $L$ neutral sites, the expectation is
$$E[k_i]\approx\frac{4N\mu L}{i}.$$

The code below simulates one population with 5000 sites. The allele absent at the start is the derived allele. Fill in the one marked line for `expected_spectrum`, then run the supplied plotting cell to compare the counts from 1 to 20.

Discuss with your partner: as the population size increases, approximately how many segregating sites in total will be observed? What fraction of segregating sites will be singletons?

In [ ]:
np.random.seed(203)
spectrum_N = 100
spectrum_mu = 1e-4
spectrum_sites = 5000
model = WrightFisher(mu=spectrum_mu, N=spectrum_N)
allele_frequency = model.evolve(np.zeros(spectrum_sites), burn_in*spectrum_N)
counts = np.round(2*spectrum_N*allele_frequency).astype(int)
i = np.arange(1, 21)
spectrum = np.array([np.sum(counts == count) for count in i])

In [ ]:
# TODO: expected number of sites with i copies of the derived allele.
expected_spectrum = ...


In [ ]:
# Supplied plotting code.
plt.figure(figsize=(8, 4.5))
plt.bar(i, spectrum[:20], color='lightgray', label='Simulated')
plt.plot(i, expected_spectrum, color='red', label=r'$4N\mu L/i$')
plt.xticks([1, 5, 10, 15, 20])
plt.xlabel('Copies of the derived allele, i')
plt.ylabel(r'Number of sites, $k_i$')
plt.legend()
plt.tight_layout()
plt.show()

<h5>Problem 5</h5>

We now have two ways of estimating $\theta=4N\mu$. Let $L$ be the number of sites, including sites with no variation:
$$\widehat\theta_H=H$$
$$\widehat\theta_S=\frac{S}{L\left(1+\frac12+\cdots+\frac{1}{2N-1}\right)}.$$

Fill in the one marked line that counts segregating sites. The heterozygosity function is already defined, and the remaining calculations are supplied.

The code below runs three conditions and plots $S$, $H$, and the ratio $\widehat\theta_H/\widehat\theta_S$. Run these cells and verify that varying $N$ and varying $\mu$ both have the expected effect on your estimates of $\theta=4N\mu$.

In [ ]:
def segregating_sites(allele_frequency):
    # TODO: count segregating sites in each row.
    S = ...
    return S

def diversity_statistics(allele_frequency, copies_observed):
    S = segregating_sites(allele_frequency)
    H = heterozygosity(allele_frequency)
    L = allele_frequency.shape[-1]
    theta_S = S/(L*sum(1/i for i in range(1, copies_observed)))
    return S, H, H/theta_S

assert segregating_sites(np.array([0, 0.1, 0.5, 1])) == 2
print('Success!')

In [ ]:
np.random.seed(204)
num_sites = 200
num_replicates = 250  # Each row gives one value for each histogram.
base_N = 50
base_mu = 1e-4
settings = [(50, 1e-4), (50, 2e-4), (100, 1e-4)]
neutral_frequencies = []
neutral_results = []
for N, mu in settings:
    model = WrightFisher(mu=mu, N=N)
    af = model.evolve(np.zeros((num_replicates, num_sites)), burn_in*N)
    neutral_frequencies.append(af)
    neutral_results.append(diversity_statistics(af, 2*N))

In [ ]:
# Supplied plotting code.
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
bins = [np.arange(0, 71, 4), np.linspace(0, 0.08, 25), np.linspace(0, 2, 25)]
print('  N       mu    Mean S    Mean H    Mean ratio')
for (N, mu), (S, H, ratio) in zip(settings, neutral_results):
    for ax, values, edges in zip(axes, [S, H, ratio], bins):
        ax.hist(values, bins=edges, density=True, histtype='step',
                linewidth=1.8, label=f'N={N}, mu={mu:g}')
    print(f'{N:3d}  {mu:7g}  {S.mean():8.2f}  {H.mean():8.4f}  {ratio.mean():10.3f}')
for ax, label in zip(axes, ['Segregating sites, S', 'Mean heterozygosity, H', 'Ratio of the estimates']):
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
axes[2].axvline(1, color='gray', linestyle='--')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

<h5>Problem 6</h5>

Tajima's D is a statistic that compares these two estimates and converts their difference into a locus-wise statistical test for departures from the neutral model. Here we will work with their ratio.

Discuss with your partner: why might it be useful to compare these two estimates? What do you expect to happen to their ratio in the presence of selection?

<h5>Problem 7</h5>

Generalize the Wright–Fisher model to include selection by filling in the one marked line:
$$p_{\mathrm{selected}}=\frac{p(1+s)}{1+ps}.$$
Mutation, drift, and the rest of the simulation are supplied. Check that the assertions in the following cell pass.

In [ ]:
class WrightFisher:
    def __init__(self, mu, N, s=0):
        self.mu = mu
        self.N = N
        self.s = s

    def next_generation(self, allele_frequencies):
        p = allele_frequencies
        # TODO: expected allele frequency after selection.
        expected_frequency = ...
        expected_frequency = self.mu + (1-2*self.mu)*expected_frequency
        num_carriers = np.random.binomial(2*self.N, expected_frequency)
        return num_carriers/(2*self.N)

    def evolve(self, allele_frequencies, generations):
        for _ in range(generations):
            allele_frequencies = self.next_generation(allele_frequencies)
        return allele_frequencies

In [ ]:
# Supplied checks: neutral and selected sites in the same population.
np.random.seed(210)
check_model = WrightFisher(mu=0, N=100, s=np.array([0, -0.5]))
check_frequencies = check_model.next_generation(np.full((2000, 2), 0.5))
assert check_frequencies.shape == (2000, 2)
assert np.all((check_frequencies >= 0) & (check_frequencies <= 1))
assert np.allclose(check_frequencies.mean(axis=0), [0.5, 1/3], atol=0.01)
print('Success!')


<h5>Problem 8</h5>

The code below simulates a population with $N=50$ and $\mu=10^{-4}$. Half of the sites are neutral and half have $s=-0.1$. It compares the ratio of the two diversity estimates with the neutral simulations from Problem 5.

Run the supplied simulation and plotting cells. Discuss with your partner: why do the selected alleles tend to be rare? What would happen if the population were smaller?

In [ ]:
np.random.seed(205)
selection_coefficients = np.zeros(num_sites)
selection_coefficients[num_sites//2:] = -0.1
model = WrightFisher(mu=base_mu, N=base_N, s=selection_coefficients)
selected_frequencies = model.evolve(np.zeros((num_replicates, num_sites)), burn_in*base_N)
selected_statistics = diversity_statistics(selected_frequencies, 2*base_N)
selection_results = [neutral_results[0], selected_statistics]
selection_labels = ['Neutral', '50% neutral, 50% s=-0.1']

In [ ]:
# Supplied plotting code.
plt.figure(figsize=(8, 4.5))
print('Model                               Mean S    Mean H    Mean ratio')
for name, (S, H, ratio) in zip(selection_labels, selection_results):
    plt.hist(ratio, bins=np.linspace(0, 2, 26), density=True,
             histtype='step', linewidth=2, label=name)
    print(f'{name:34s} {S.mean():7.2f}  {H.mean():8.4f}  {ratio.mean():10.3f}')
plt.axvline(1, color='gray', linestyle='--')
plt.xlabel('Ratio of the estimates')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

<h5>Problem 9</h5>

A limitation of the diversity comparison is that it does not directly estimate a selection coefficient.

Recall that $s_{het}$ is the selection coefficient of protein truncating variants (PTVs) in a gene.

Suppose that you have genetic data for a population of $N=100$ individuals. There are $L=500$ sites at which a mutation could create a PTV (possibly, not all of which are observed in your data), each of which has a mutation rate of $\mu=4\times10^{-4}$. ($L\mu$ is high in this example to compensate for small $N$). All true PTVs in the gene have the same selection coefficient, $s_{het}$, which is unknown. Using simulations, you will estimate $s_{het}$ from mutation–selection balance.

For a rare deleterious allele, $p_j\approx-\mu_j/s_{het}$, so
$$\widehat s_{het}=-\frac{\sum_{j=1}^L\mu_j}{\sum_{j=1}^L p_j}.$$
As in lecture, a deleterious effect has $s_{het}<0$, although you will often encounter the opposite convention. The mutation rates in the numerator include every site at which a PTV could occur, including sites where none is observed.

Fill in the one marked line for this estimator. The supplied code uses the Wright–Fisher model from Problem 7 to simulate 100 genes for each of $s_{het}\in\{-0.05,-0.1,-0.2\}$. It starts with every frequency equal to zero and runs for $4N=400$ generations. Run the supplied simulation and plotting cells, then compare the estimates with the true values.

In [ ]:
def estimate_s_het(allele_frequency, mu):
    L = allele_frequency.shape[-1]
    burden = np.sum(allele_frequency, axis=-1)
    # TODO: mutation–selection balance estimate.
    estimate = ...
    return estimate

In [ ]:
np.random.seed(206)
gene_N = 100
gene_L = 500
gene_mu = 4e-4
num_genes = 100
s_values = [-0.05, -0.1, -0.2]
gene_populations = {}
gene_results = {}
print('True s    Median estimate    Pooled estimate')
for s in s_values:
    model = WrightFisher(mu=gene_mu, N=gene_N, s=s)
    af = model.evolve(np.zeros((num_genes, gene_L)), burn_in*gene_N)
    gene_populations[s] = af
    estimates = estimate_s_het(af, gene_mu)
    gene_results[s] = estimates
    pooled = -gene_L*gene_mu/af.sum(axis=1).mean()
    print(f'{s:6.2f}    {np.median(estimates):14.4f}    {pooled:15.4f}')

In [ ]:
# Supplied plotting code.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, s in zip(axes, s_values):
    estimates = gene_results[s]
    ax.hist(estimates, bins=20, color='lightgray', edgecolor='white')
    ax.axvline(s, color='red', linestyle='--', label='True value')
    ax.set_xlabel(r'Estimated $s_{het}$')
    ax.set_title(f'True s = {s:g}')
    ax.locator_params(axis='x', nbins=4)
    ax.legend()
axes[0].set_ylabel('Number of genes')
plt.tight_layout()
plt.show()

<h5>Problem 10</h5>

Discuss with your partner: what happens if a small fraction of sites are false positives—annotated as PTVs, but actually neutral? What do you expect to happen to the estimate of $s_{het}$? Does it matter how strongly selected the true PTVs are?

The code below tests this with $N=50$, $L=1000$, $\mu=2\times10^{-4}$, and $s_{het}=-0.1$ or $-0.4$. It sets 0%, 0.1%, 0.5%, or 1% of the annotated sites to be neutral and keeps all sites in the estimator, including the false positives.

Fill in the one marked line specifying the selection coefficient of a false positive. The supplied code simulates all sites together, starting at zero and running for $4N=200$ generations. Run the simulation and plotting cells, then compare each result with the 0% false-positive condition.

In [ ]:
np.random.seed(207)
annotation_N = 50
annotation_L = 1000
annotation_mu = 2e-4
annotation_num_genes = 100
annotation_s_values = [-0.1, -0.4]
false_positive_fractions = np.array([0, 0.001, 0.005, 0.01])
false_positive_results = {}
print('True s   Neutral fraction   Median estimate   Pooled estimate')
for s in annotation_s_values:
    for fraction in false_positive_fractions:
        selection_coefficients = np.full(annotation_L, s)
        count = round(fraction*annotation_L)
        # TODO: selection coefficient of sites that are actually neutral.
        selection_coefficients[:count] = ...
        model = WrightFisher(mu=annotation_mu, N=annotation_N, s=selection_coefficients)
        af = model.evolve(np.zeros((annotation_num_genes, annotation_L)), burn_in*annotation_N)
        estimates = estimate_s_het(af, annotation_mu)
        pooled = -annotation_L*annotation_mu/af.sum(axis=1).mean()
        false_positive_results[(s, fraction)] = dict(estimates=estimates, pooled=pooled)
        print(f'{s:6.2f}  {fraction:17.3f}  {np.median(estimates):16.4f}  {pooled:16.4f}')

In [ ]:
# Supplied plotting code.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, s in zip(axes, annotation_s_values):
    results = [false_positive_results[(s, fraction)] for fraction in false_positive_fractions]
    medians = [np.median(r['estimates']) for r in results]
    lower = [np.quantile(r['estimates'], 0.1) for r in results]
    upper = [np.quantile(r['estimates'], 0.9) for r in results]
    pooled = [r['pooled'] for r in results]
    x = 100*false_positive_fractions
    ax.fill_between(x, lower, upper, alpha=0.15, label='10th–90th percentiles')
    ax.plot(x, medians, 'o-', label='Median estimate')
    ax.plot(x, pooled, 's--', label='Pooled estimate')
    ax.axhline(s, color='red', linestyle=':', label='True value')
    ax.set_ylim(min(1.15*s, min(lower))*1.05, 0)
    ax.set_xlabel('Neutral sites among annotated PTVs (%)')
    ax.set_ylabel(r'Estimated $s_{het}$')
    ax.set_title(f'True s = {s:g}')
axes[0].legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

<h5>Problem 11</h5>

Discuss with your partner: what do you expect population growth to do to the ratio $\widehat\theta_H/\widehat\theta_S$?

**Hint:** think about the coalescent. Does an expanding population enrich for rare alleles, common alleles, or neither?

The code below continues the neutral simulations from Problem 5 for another 100 generations. One population stays at $N=50$; the other grows to $N=500$. Fill in the marked line that updates the population size each generation. Run the supplied plotting cell to compare the ratio for the two populations.

In [ ]:
np.random.seed(208)
initial_frequencies = neutral_frequencies[0]
constant_model = WrightFisher(mu=base_mu, N=base_N)
constant_frequencies = constant_model.evolve(initial_frequencies.copy(), 100)

model = WrightFisher(mu=base_mu, N=base_N)
growing_frequencies = initial_frequencies.copy()
for t in range(1, 101):
    # TODO: increase the population size tenfold over 100 generations.
    model.N = ...
    growing_frequencies = model.next_generation(growing_frequencies)

constant_ratio = diversity_statistics(constant_frequencies, 2*base_N)[2]
growing_ratio = diversity_statistics(growing_frequencies, 2*model.N)[2]

In [ ]:
# Supplied plotting code.
plt.figure(figsize=(8, 4.5))
for label, ratio in [('Constant size', constant_ratio), ('Growth', growing_ratio)]:
    plt.hist(ratio, bins=np.linspace(0, 2, 26), density=True,
             histtype='step', linewidth=2, label=label)
    print(f'{label}: mean ratio = {ratio.mean():.3f}')
plt.axvline(1, color='gray', linestyle='--')
plt.xlabel(r'Ratio of the estimates, $\widehat\theta_H/\widehat\theta_S$')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

<h5>Problem 12</h5>

What do you expect population growth to do to the mutation–selection balance estimate?

The code below does the same experiment using the PTV simulations from Problem 9, with $s_{het}=-0.1$. One population stays at $N=100$; the other grows to $N=1000$ over 100 generations. Fill in the marked line and run the plots. Compare the number of segregating sites and the estimate of $s_{het}$ between the two populations.

In [ ]:
np.random.seed(209)
s = -0.1
initial_frequencies = gene_populations[s]
constant_model = WrightFisher(mu=gene_mu, N=gene_N, s=s)
constant_frequencies = constant_model.evolve(initial_frequencies.copy(), 100)

model = WrightFisher(mu=gene_mu, N=gene_N, s=s)
growing_frequencies = initial_frequencies.copy()
for t in range(1, 101):
    # TODO: increase the population size tenfold over 100 generations.
    model.N = ...
    growing_frequencies = model.next_generation(growing_frequencies)

gene_growth_results = {}
for label, af in [('Constant size', constant_frequencies), ('Growth', growing_frequencies)]:
    S = segregating_sites(af)
    estimates = estimate_s_het(af, gene_mu)
    gene_growth_results[label] = (S, estimates)
    print(f'{label}: mean S = {S.mean():.1f}, median s = {np.median(estimates):.3f}')

In [ ]:
# Supplied plotting code.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for label, (S, estimates) in gene_growth_results.items():
    axes[0].hist(S, bins=np.arange(0, gene_L+21, 20), density=True,
                 histtype='step', linewidth=2, label=label)
    axes[1].hist(estimates, bins=np.linspace(-0.16, -0.05, 23), density=True,
                 histtype='step', linewidth=2, label=label)
axes[0].set_xlabel('Segregating PTV sites')
axes[1].set_xlabel(r'Estimated $s_{het}$')
axes[1].axvline(s, color='gray', linestyle='--', label='True value')
for ax in axes:
    ax.set_ylabel('Density')
    ax.legend()
plt.tight_layout()
plt.show()

<h5>References</h5>

Tajima, F. (1989). [Statistical method for testing the neutral mutation hypothesis by DNA polymorphism](https://doi.org/10.1093/genetics/123.3.585). *Genetics* 123:585–595.

Simons, Y. B., Turchin, M. C., Pritchard, J. K. and Sella, G. (2014). [The deleterious mutation load is insensitive to recent population history](https://doi.org/10.1038/ng.2896). *Nature Genetics* 46:220–224.